In [93]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import random
import mlflow

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score, f1_score, brier_score_loss, auc
from sklearn.preprocessing import StandardScaler, label_binarize, LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from mlflow.tracking import MlflowClient
from scipy.stats import circmean

import xgboost as xgb

from itertools import product

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1000)

In [94]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("btc_timezone_analysis_sensitivity_analysis")

<Experiment: artifact_location='/home/matej/btc_timezone_analysis/model_training/mlruns_artifacts/3', creation_time=1790779292325, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1790779292325, lifecycle_stage='active', name='btc_timezone_analysis_sensitivity_analysis', tags={}, trace_location=None, workspace='default'>

In [95]:
def calculate_aurc(y_true, y_prob, y_pred):
    """
    Calculates the Area Under the Risk-Coverage Curve (AURC).
    """
    # 1. Convert to pure NumPy arrays to avoid Pandas KeyErrors
    y_true_np = np.array(y_true)
    y_pred_np = np.array(y_pred)
    
    # 2. Extract the model's confidence
    confidences = np.max(y_prob, axis=1)
    
    # 3. Determine which predictions are incorrect (Error = 1, Correct = 0)
    # By using y_pred directly, this safely compares strings to strings or ints to ints!
    errors = (y_true_np != y_pred_np).astype(int)
    
    # 4. Sort the samples by confidence in descending order
    sorted_indices = np.argsort(-confidences)
    sorted_errors = errors[sorted_indices]
    
    # 5. Calculate Coverage (x-axis)
    n_samples = len(y_true_np)
    coverages = np.arange(1, n_samples + 1) / n_samples
    
    # 6. Calculate Risk (y-axis): The cumulative mean of errors
    cumulative_errors = np.cumsum(sorted_errors)
    risks = cumulative_errors / np.arange(1, n_samples + 1)
    
    # 7. Add the starting point (0 coverage) to anchor the graph
    coverages = np.insert(coverages, 0, 0.0)
    risks = np.insert(risks, 0, risks[0]) 
    
    # 8. Calculate the area under the curve using the trapezoidal rule
    aurc_score = auc(coverages, risks)
    
    return coverages, risks, aurc_score

In [96]:
def get_mlflow_confidence_metrics(
    y_val_true, y_val_pred, y_val_prob, 
    y_train_true, y_train_pred, y_train_prob, 
    target_names
):
    metrics = {}

    # 1. Standard Metrics
    metrics['train_brier_score'] = float(brier_score_loss(label_binarize(y_train_true, classes=target_names), y_train_prob))
    metrics['val_brier_score'] = float(brier_score_loss(label_binarize(y_val_true, classes=target_names), y_val_prob))
    metrics['train_accuracy'] = float(accuracy_score(y_train_true, y_train_pred))
    metrics['val_accuracy'] = float(accuracy_score(y_val_true, y_val_pred))
    metrics['samples'] = len(y_val_true)

    fold_metrics_collection = {
        "val_coverage_10_accuracy": [], 
        "val_coverage_25_accuracy": [], 
        "val_coverage_50_accuracy": [], 
        "val_aurc": [],
        "train_aurc": []
    }
    
    n_splits = 5
    
    # Calculate separate fold sizes because the concatenated train array 
    # is 4x larger than the validation array
    val_fold_size = len(y_val_true) // n_splits
    train_fold_size = len(y_train_true) // n_splits
    
    for i in range(n_splits):
        # 1. Validation Slice
        val_start = i * val_fold_size
        val_end = (i + 1) * val_fold_size if i < n_splits - 1 else len(y_val_true)
        
        y_val_true_fold = np.array(y_val_true[val_start:val_end])
        y_val_pred_fold = np.array(y_val_pred[val_start:val_end])
        y_val_prob_fold = np.array(y_val_prob[val_start:val_end])
        
        # 2. Train Slice
        train_start = i * train_fold_size
        train_end = (i + 1) * train_fold_size if i < n_splits - 1 else len(y_train_true)
        
        y_train_true_fold = np.array(y_train_true[train_start:train_end])
        y_train_pred_fold = np.array(y_train_pred[train_start:train_end])
        y_train_prob_fold = np.array(y_train_prob[train_start:train_end])

        # 3. Validation Coverage Metrics
        confidences = np.max(y_val_prob_fold, axis=1)
        sorted_idx = np.argsort(confidences)[::-1]
        y_true_sorted = y_val_true_fold[sorted_idx]
        y_pred_sorted = y_val_pred_fold[sorted_idx]
        
        coverages = [0.10, 0.25, 0.50]
        for c in coverages:
            k = max(1, int(c * len(y_val_true_fold))) 
            top_true = y_true_sorted[:k]
            top_pred = y_pred_sorted[:k]
            
            c_str = f"val_coverage_{int(c * 100)}"
            fold_metrics_collection[f"{c_str}_accuracy"].append(float(accuracy_score(top_true, top_pred)))
            
        # 4. Validation AURC Integration
        _, _, val_fold_aurc = calculate_aurc(y_val_true_fold, y_val_prob_fold, y_val_pred_fold)
        fold_metrics_collection['val_aurc'].append(float(val_fold_aurc))
        
        # 5. Train AURC Integration
        _, _, train_fold_aurc = calculate_aurc(y_train_true_fold, y_train_prob_fold, y_train_pred_fold)
        fold_metrics_collection['train_aurc'].append(float(train_fold_aurc))
        
    # Average all metrics across the 5 folds
    for key, values in fold_metrics_collection.items():
        metrics[key] = float(np.mean(values))
    
    return metrics

In [97]:
#Importing data from csv, the path could be different
data = pd.read_csv("/home/matej/btc_timezone_analysis/data_preprocessing/database_from_bitcointalk.csv")
print(len(data))
data.columns = data.columns.astype(str)
data.head()

1936


,Unnamed: 0,entity,offset,is_regional_location,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,total,log_total,min_5h_activity,min_5h_activity_sin,min_5h_activity_cos,min_5h_activity_%,lifespan_days,0_weekday,1_weekday,2_weekday,3_weekday,4_weekday,5_weekday,6_weekday,7_weekday,8_weekday,9_weekday,10_weekday,11_weekday,12_weekday,13_weekday,14_weekday,15_weekday,16_weekday,17_weekday,18_weekday,19_weekday,20_weekday,21_weekday,22_weekday,23_weekday,0_weekend,1_weekend,2_weekend,3_weekend,4_weekend,5_weekend,6_weekend,7_weekend,8_weekend,9_weekend,10_weekend,11_weekend,12_weekend,13_weekend,14_weekend,15_weekend,16_weekend,17_weekend,18_weekend,19_weekend,20_weekend,21_weekend,22_weekend,23_weekend,total_weekday,total_weekend,dst_shift,region
0,0,Entity 100,7.0,0,0.020921,0.037657,0.025105,0.050209,0.050209,0.058577,0.033473,0.054393,0.050209,0.046025,0.046025,0.071130,0.037657,0.033473,0.054393,0.054393,0.062762,0.112971,0.037657,0.016736,0.012552,0.016736,0.004184,0.012552,215,5.370638,21,-0.707107,0.707107,0.062762,527.561829,0.022599,0.050847,0.028249,0.050847,0.067797,0.073446,0.039548,0.045198,0.033898,0.056497,0.050847,0.050847,0.028249,0.022599,0.056497,0.073446,0.084746,0.062147,0.033898,0.022599,0.011299,0.016949,0.005650,0.011299,0.023256,0.011628,0.023256,0.046512,0.011628,0.023256,0.023256,0.069767,0.081395,0.023256,0.034884,0.104651,0.058140,0.058140,0.046512,0.011628,0.011628,0.197674,0.046512,0.011628,0.023256,0.023256,0.011628,0.023256,153.0,62.0,0.0,East_Asia_Pac
1,1,Entity 1000,2.0,0,0.107143,0.107143,0.071429,0.035714,0.017857,0.035714,0.053571,0.017857,0.035714,0.035714,0.089286,0.035714,0.053571,0.053571,0.017857,0.017857,0.017857,0.035714,0.017857,0.035714,0.017857,0.035714,0.017857,0.035714,32,3.465736,16,-0.866025,-0.500000,0.107143,125.700324,0.100000,0.100000,0.080000,0.040000,0.020000,0.020000,0.040000,0.020000,0.040000,0.040000,0.080000,0.040000,0.060000,0.040000,0.020000,0.020000,0.020000,0.040000,0.020000,0.040000,0.020000,0.040000,0.020000,0.040000,0.066667,0.066667,0.033333,0.033333,0.033333,0.066667,0.066667,0.033333,0.033333,0.033333,0.066667,0.033333,0.033333,0.066667,0.033333,0.033333,0.033333,0.033333,0.033333,0.033333,0.033333,0.033333,0.033333,0.033333,26.0,6.0,11.0,Euro_Africa
2,2,Entity 1001,2.0,0,0.034483,0.034483,0.051724,0.051724,0.068966,0.068966,0.103448,0.172414,0.051724,0.034483,0.034483,0.034483,0.017241,0.051724,0.017241,0.017241,0.017241,0.034483,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,34,3.526361,20,-0.866025,0.500000,0.086207,1093.023981,0.035714,0.035714,0.035714,0.053571,0.071429,0.071429,0.107143,0.178571,0.053571,0.035714,0.035714,0.035714,0.017857,0.053571,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.038462,0.038462,0.076923,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,0.076923,0.038462,0.038462,0.038462,0.038462,0.038462,0.038462,32.0,2.0,-4.0,Euro_Africa
3,3,Entity 1002,-5.0,0,0.053571,0.071429,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.017857,0.035714,0.035714,0.053571,0.071429,0.125000,0.035714,0.053571,0.017857,0.071429,0.053571,0.035714,0.107143,32,3.465736,4,0.866025,0.500000,0.089286,1229.220995,0.066667,0.066667,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.022222,0.044444,0.088889,0.133333,0.044444,0.066667,0.022222,0.066667,0.066667,0.022222,0.044444,0.028571,0.057143,0.028571,0.028571,0.028571,0.028571,0.028571,0.028571,0.028571,0.028571,0.028571,0.028571,0.057143,0.057143,0.057143,0.028571,0.057143,0.028571,0.028571,0.028571,0.057143,0.028571,0.057143,0.142857,21.0,11.0,2.0,America
4,5,Entity 1004,7.0,0,0.017241,0.017241,0.017241,0.051724,0.086207,0.068966,0.086207,0.051724,0.017241,0.034483,0.086207,0.017241,0.103448,0.051724,0.017241,0.017241,0.051724,0.034483,0.068966,0.017241,0.034483,0.017241,0.017241,0.01

In [98]:
X = data
y = data["region"]

col_names = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20", "21", "22", "23", "dst_shift"]

In [99]:
num = 1019
print(num)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify = y, random_state = num)

#Creating sample weights, which put higher weight on smaller classes and lower weight on bigger classes 
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

1019


In [100]:
with mlflow.start_run(run_name = "xgboost_model"):
    target_names = ["America", "Euro_Africa", "Central_South_Asia", "East_Asia_Pac"]
    colors = ['red', 'green', 'blue', 'black']
    
    lambda_par = 0.5  # You can adjust this value based on how much you want to penalize overfitting
    experiment_name = f"xgboost hyperparameter fitting with lambda={lambda_par} with higher min eta and higher max gamma"
    mlflow.set_tag("model_type", experiment_name)
    mlflow.set_tag("exclusion_num_txs_min", 15)
    mlflow.set_tag("exclusion_num_txs_max", "inf")
    mlflow.set_tag("data type", "24 hour transaction distribution")

    kf = StratifiedKFold(n_splits=5, shuffle=True)

    y_val_cv = []
    y_val_cv_pred = []
    y_val_cv_proba = []

    y_train_cv = []
    y_train_cv_pred = []
    y_train_cv_proba = []

    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    pipeline = xgb.XGBClassifier(
        n_estimators = 777,
        eta = 0.05564936741180356,
        gamma = 5.724469359582323,
        reg_lambda = 1.0005545286124224,
        max_depth = 5,
        min_child_weight = 13,
        subsample = 0.6649481827307877,
        colsample_bytree = 0.9093752666296373,
        random_state = 1019
    )

    for train_idx, val_idx in kf.split(X_train[col_names], y_train_encoded):
        X_tr, X_va = X_train[col_names].iloc[train_idx], X_train[col_names].iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
        y_tr, y_va = y_train_encoded[train_idx], y_train_encoded[val_idx]
        # Train the pipeline
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_tr)
        pipeline.fit(X_tr, y_tr, sample_weight = sample_weights)

        # Predict and store Training data
        y_train_cv.extend(le.inverse_transform(y_tr))
        y_train_cv_pred.extend(le.inverse_transform(pipeline.predict(X_tr)))
        y_train_cv_proba.extend(pipeline.predict_proba(X_tr))
        
        # Predict and store Validation data
        y_val_cv.extend(le.inverse_transform(y_va))
        y_val_cv_pred.extend(le.inverse_transform(pipeline.predict(X_va)))
        y_val_cv_proba.extend(pipeline.predict_proba(X_va))

    # Convert training lists to numpy arrays
    y_train_cv = np.array(y_train_cv)
    y_train_cv_pred = np.array(y_train_cv_pred)
    y_train_cv_proba = np.array(y_train_cv_proba)
    y_val_cv = np.array(y_val_cv)
    y_val_cv_pred = np.array(y_val_cv_pred)
    y_val_cv_proba = np.array(y_val_cv_proba)

    metrics = get_mlflow_confidence_metrics(
        y_val_cv, y_val_cv_pred, y_val_cv_proba, 
        y_train_cv, y_train_cv_pred, y_train_cv_proba, 
        le.inverse_transform(pipeline.classes_)
    )
    mlflow.log_param("lambda", lambda_par)
    mlflow.log_param("random_state train - test split", num)
    mlflow.log_metrics(metrics)
        
    # Overfitting Penalty Optimization Score
    final_score = metrics["val_brier_score"] + lambda_par * np.abs(metrics["val_brier_score"] - metrics["train_brier_score"])
    
    mlflow.log_metric("optuna_penalized_score", final_score)
        

🏃 View run xgboost_model at: http://127.0.0.1:5000/#/experiments/3/runs/0de86d2a5bec4e47b866ff1934d7c3da
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
